##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# 1) Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

# Convert images to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

c:\Users\saras\anaconda3\envs\cv_lab\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [2]:
# 2) Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

# 3) Build MobileNetV2 backbone (pretrained)
mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
mobilenet_base.trainable = False  # Freeze backbone initially

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# 4) Full model (Preprocessing inside)
mobilenet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input), 
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(10)  # Logits for 10 classes
], name="cifar10_mobilenetv2")

mobilenet_model.summary()

Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
# 5) Compile + Train (frozen backbone)
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

print("Starting Initial Training...")
history = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

# Evaluate Frozen Performance
test_loss, test_acc_m = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print(f"\nMobileNetV2 (frozen) test accuracy: {test_acc_m:.4f}")

Starting Initial Training...
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1085s 2s/step - accuracy: 0.6539 - loss: 0.9955 - val_accuracy: 0.8054 - val_loss: 0.5585
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1101s 2s/step - accuracy: 0.7188 - loss: 0.8013 - val_accuracy: 0.8102 - val_loss: 0.5653
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1057s 2s/step - accuracy: 0.7325 - loss: 0.7708 - val_accuracy: 0.8188 - val_loss: 0.5273

MobileNetV2 (frozen) test accuracy: 0.8106


In [5]:
# 6) Fine-tune last layers
mobilenet_base.trainable = True

# Freeze all layers EXCEPT the last 20
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

print(f"Trainable layers in backbone: {sum(l.trainable for l in mobilenet_base.layers)} / {len(mobilenet_base.layers)}")

# Re-compile with a LOWER learning rate for fine-tuning
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

print("Starting Fine-tuning...")
history_ft = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

# Final Evaluation
test_loss_ft, test_acc_ft = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print(f"\nMobileNetV2 (fine-tuned) test accuracy: {test_acc_ft:.4f}")

Trainable layers in backbone: 20 / 154
Starting Fine-tuning...
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1393s 2s/step - accuracy: 0.6976 - loss: 0.8800 - val_accuracy: 0.8360 - val_loss: 0.4836
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1445s 2s/step - accuracy: 0.7497 - loss: 0.7233 - val_accuracy: 0.8416 - val_loss: 0.4511
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1418s 2s/step - accuracy: 0.7696 - loss: 0.6592 - val_accuracy: 0.8478 - val_loss: 0.4288

MobileNetV2 (fine-tuned) test accuracy: 0.8422


1. Which model achieved the highest accuracy?

ResNet50V2 achieved the highest accuracy at 91.62% after fine-tuning, followed by MobileNetV2 which reached 84.22% also through fine-tuning. The Custom CNN came last at 70.04%. This confirms that fine-tuning pretrained models is much more effective than training a model from scratch.






2. Which model trained faster?

MobileNetV2 trained significantly faster than ResNet50V2 because it is designed for efficiency and has fewer parameters. Although the Custom CNN was fast due to its simple structure, it failed to reach a high accuracy level compared to the others.





3. How might the architecture explain the differences?



ResNet50V2: Its high accuracy comes from its depth and Residual Connections, which allow it to learn very complex features, but also make it a "heavy" model with about 23.5 million parameters.

MobileNetV2: It is designed for efficiency using Depthwise Separable Convolutions. This reduces its size to only 2.2 million parameters, explaining why it is much faster and lighter than ResNet while still maintaining good accuracy.

Custom CNN: It lacks the advanced layers and the "pretrained knowledge" of the other models. Since it starts learning from scratch with a basic structure, it cannot match the feature extraction power of the global architectures.